# Faruq-v3 — AF2 × D-FINE-N seed-42 transfer screen

Paired validation-only experiment: `DFN0` vs `frozen AF2 + D-FINE-N`. Locked test remains closed.

**Environment is frozen before the experiment.** Kaggle's preinstalled PyTorch can change and may omit Pascal `sm_60`; this notebook pins `torch 2.5.1+cu124` + `torchvision 0.20.1`, verifies the allocated GPU architecture is compiled into PyTorch, and executes a real CUDA tensor smoke-test before cloning/training.

The v2 runner also enforces the corrected D-FINE custom-dataset label contract `0..20` when `remap_mscoco_category=False`.

Required Kaggle input: exactly one `faruq-development-v3-grouped.tar.bin`. **Do not attach a state ZIP from a failed run.** GPU and Internet must be enabled.


In [ ]:
from pathlib import Path
import json, subprocess, sys

WORK=Path('/kaggle/working')
if not WORK.is_dir():
    raise RuntimeError('Kaggle-only notebook')

PIN_TORCH='2.5.1'
PIN_TORCHVISION='0.20.1'
TORCH_INDEX='https://download.pytorch.org/whl/cu124'

def torch_probe():
    code = r'''import json, importlib.metadata as md
try:
    import torch
    payload={'ok':True,'torch':torch.__version__,'torchvision':md.version('torchvision'),'cuda_runtime':torch.version.cuda,'cuda_available':torch.cuda.is_available(),'arch_list':torch.cuda.get_arch_list()}
    if torch.cuda.is_available():
        payload['gpu']=torch.cuda.get_device_name(0)
        payload['cc']=list(torch.cuda.get_device_capability(0))
    print(json.dumps(payload))
except Exception as exc:
    print(json.dumps({'ok':False,'error':repr(exc)}))'''
    return json.loads(subprocess.check_output([sys.executable,'-c',code],text=True).strip().splitlines()[-1])

before=torch_probe()
print('TORCH BEFORE:',json.dumps(before,indent=2))
if not before.get('cuda_available'):
    raise RuntimeError('Aktifkan Kaggle GPU sebelum menjalankan notebook')
required_arch=f"sm_{before['cc'][0]}{before['cc'][1]}"
needs_pin=(not before.get('ok') or not str(before.get('torch','')).startswith('2.5.1+cu124') or str(before.get('torchvision','')).split('+')[0] != PIN_TORCHVISION or required_arch not in before.get('arch_list',[]))
if needs_pin:
    print(f'PIN PYTORCH FOR {before.get("gpu")} / {required_arch} ...')
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-cache-dir','--upgrade',f'torch=={PIN_TORCH}',f'torchvision=={PIN_TORCHVISION}','--index-url',TORCH_INDEX],check=True)

after=torch_probe()
print('TORCH FROZEN:',json.dumps(after,indent=2))
if not after.get('cuda_available'):
    raise RuntimeError('CUDA unavailable after PyTorch pin')
required_arch=f"sm_{after['cc'][0]}{after['cc'][1]}"
if not str(after.get('torch','')).startswith('2.5.1+cu124'):
    raise RuntimeError(f"Unexpected torch after pin: {after.get('torch')}")
if required_arch not in after.get('arch_list',[]):
    raise RuntimeError(f"GPU {after.get('gpu')} requires {required_arch}, but torch provides {after.get('arch_list')}")
smoke=r'''import torch
x=torch.arange(4096,dtype=torch.float32,device='cuda').reshape(64,64)
y=(x @ x.T).mean()
torch.cuda.synchronize()
assert torch.isfinite(y).item()
print('CUDA_SMOKE_PASS',torch.__version__,torch.cuda.get_device_name(0),torch.cuda.get_device_capability(0),float(y))'''
subprocess.run([sys.executable,'-c',smoke],check=True)


In [ ]:
import shutil, subprocess, sys

COFFEE=WORK/'coffee-bean-detection'
DFINE=WORK/'D-FINE'
BRANCH='agent/af2-dfine-n-transfer-screen'
DFINE_COMMIT='956d1709314c2c6a4df6f34de232054578a7449f'

for path in (COFFEE,DFINE):
    if path.exists():
        shutil.rmtree(path)

subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(COFFEE)],check=True)
subprocess.run(['git','clone','--depth','1','https://github.com/Peterande/D-FINE.git',str(DFINE)],check=True)
head=subprocess.check_output(['git','rev-parse','HEAD'],cwd=DFINE,text=True).strip()
if head!=DFINE_COMMIT:
    subprocess.run(['git','fetch','origin',DFINE_COMMIT,'--depth','1'],cwd=DFINE,check=True)
    subprocess.run(['git','checkout','--detach',DFINE_COMMIT],cwd=DFINE,check=True)
head=subprocess.check_output(['git','rev-parse','HEAD'],cwd=DFINE,text=True).strip()
if head!=DFINE_COMMIT:
    raise RuntimeError(f'D-FINE commit mismatch: {head}')

subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(COFFEE)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','faster-coco-eval>=1.6.6','tensorboard','scipy','calflops','transformers','loguru','PyYAML'],check=True)
print('COFFEE:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=COFFEE,text=True).strip())
print('D-FINE:',head)


In [ ]:
subprocess.run([sys.executable,'-u',str(COFFEE/'scripts/run_dfine_af2_kaggle_screen_v3.py')],cwd=COFFEE,check=True)
